# Phase 3: Feature Engineering — Team Level

**Goal:** For every match in the dataset, compute ~24 features using ONLY data available BEFORE that match date.

**The golden rule:** Features for a match on date D must only use matches with date < D.
Violating this = data leakage = fake accuracy.

**Features we build:**
- Historical win rates (overall, venue, head-to-head, bat first)
- Recent form (last 5 matches)
- Win streak
- Toss & venue features
- Season context
- Conditions (day/night, venue avg score)

**Output:** `data/processed/team_features.csv` — one row per match, 24 features + target

## 1. Imports & Load

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

balls   = pd.read_csv('../data/processed/cleaned_balls.csv', parse_dates=['date'], low_memory=False)
matches = pd.read_csv('../data/processed/cleaned_matches.csv', parse_dates=['date'])

print(f'Balls  : {len(balls):,} rows')
print(f'Matches: {len(matches):,} rows')
matches.head(3)

## 2. Understand the Match Structure

Each match has TWO teams. We need to label them consistently as `team_a` and `team_b`.
**Convention:** team_a = the team that batted FIRST in that match.

In [ ]:
# Get the team that batted first in each match (innings == 1)
# This gives us a clean team_a / team_b assignment per match
innings1 = (
    balls[balls['innings'] == 1]
    [['match_id', 'batting_team', 'bowling_team']]
    .drop_duplicates(subset='match_id')
    .rename(columns={'batting_team': 'team_a', 'bowling_team': 'team_b'})
)

# Merge into matches
matches = matches.merge(innings1, on='match_id', how='left')

# Create target: 1 if team_a won, 0 if team_b won
matches['team_a_won'] = (matches['winner'] == matches['team_a']).astype(int)

print(f'Matches with team assignments: {matches["team_a"].notna().sum()}')
print(f'team_a win rate: {matches["team_a_won"].mean():.3f} (should be ~0.5)')
matches[['match_id','date','team_a','team_b','winner','team_a_won']].head()

## 3. Helper — Safe Division

Many features are rates (wins / matches). If a team has 0 matches at a venue,
we'd divide by zero. This helper returns a default value instead.

In [ ]:
def safe_div(numerator, denominator, default=0.5):
    """Divide two numbers safely. Returns default if denominator is 0."""
    if denominator == 0:
        return default
    return numerator / denominator

## 4. Build Features Row by Row

For each match, we look at all PAST matches (date < current match date) and compute features.

**Why row by row?** Because each match has a different cutoff date — we can't use a single
groupby. We need "CSK's win rate as of April 10, 2023" which changes match by match.

In [ ]:
# Sort matches chronologically — important for the loop
matches = matches.sort_values('date').reset_index(drop=True)

print(f'Date range: {matches["date"].min().date()} → {matches["date"].max().date()}')
print(f'Total matches to process: {len(matches)}')

In [ ]:
def compute_team_features(match_row, past_matches):
    """
    Compute all team-level features for one match using only past match data.
    
    match_row  : one row from matches DataFrame (the match we're predicting)
    past_matches: all matches with date strictly BEFORE match_row['date']
    """
    ta = match_row['team_a']   # team that batted first
    tb = match_row['team_b']   # team that bowled first
    venue = match_row['venue']
    season = match_row['season_year']

    # ── Helper: get all matches involving a team ──────────────────────────
    def team_matches(team):
        return past_matches[
            (past_matches['team_a'] == team) | (past_matches['team_b'] == team)
        ]

    def team_wins(team, df):
        return (df['winner'] == team).sum()

    # ── 1-2: Overall win rates ────────────────────────────────────────────
    ta_all = team_matches(ta)
    tb_all = team_matches(tb)
    ta_overall_wr = safe_div(team_wins(ta, ta_all), len(ta_all))
    tb_overall_wr = safe_div(team_wins(tb, tb_all), len(tb_all))

    # ── 3-4: Venue win rates ──────────────────────────────────────────────
    ta_venue = ta_all[ta_all['venue'] == venue]
    tb_venue = tb_all[tb_all['venue'] == venue]
    ta_venue_wr = safe_div(team_wins(ta, ta_venue), len(ta_venue))
    tb_venue_wr = safe_div(team_wins(tb, tb_venue), len(tb_venue))

    # ── 5: Head-to-head win rate for team_a ───────────────────────────────
    h2h = past_matches[
        ((past_matches['team_a'] == ta) & (past_matches['team_b'] == tb)) |
        ((past_matches['team_a'] == tb) & (past_matches['team_b'] == ta))
    ]
    ta_h2h_wr = safe_div(team_wins(ta, h2h), len(h2h))

    # ── 6-7: Bat-first win rates ──────────────────────────────────────────
    # team_a batted first in their past matches
    ta_batfirst = past_matches[past_matches['team_a'] == ta]  # team_a = batted first
    tb_batfirst = past_matches[past_matches['team_a'] == tb]
    ta_batfirst_wr = safe_div(team_wins(ta, ta_batfirst), len(ta_batfirst))
    tb_batfirst_wr = safe_div(team_wins(tb, tb_batfirst), len(tb_batfirst))

    # ── 8-9: Last 5 matches win rate ──────────────────────────────────────
    ta_last5 = ta_all.tail(5)
    tb_last5 = tb_all.tail(5)
    ta_last5_wr = safe_div(team_wins(ta, ta_last5), len(ta_last5))
    tb_last5_wr = safe_div(team_wins(tb, tb_last5), len(tb_last5))

    # ── 10-11: Last 5 matches average win margin ──────────────────────────
    # win_outcome examples: '45 runs', '6 wickets' — extract the number
    def avg_margin(team, df):
        wins = df[df['winner'] == team]['win_outcome'].dropna()
        margins = wins.str.extract(r'(\d+)')[0].astype(float)
        return margins.mean() if len(margins) > 0 else 0

    ta_last5_margin = avg_margin(ta, ta_last5)
    tb_last5_margin = avg_margin(tb, tb_last5)

    # ── 12-13: Win streak ─────────────────────────────────────────────────
    def win_streak(team, df):
        """Count consecutive wins at the END of df (most recent matches)."""
        streak = 0
        for _, row in df.iloc[::-1].iterrows():   # iterate in reverse (most recent first)
            if row['winner'] == team:
                streak += 1
            else:
                break
        return streak

    ta_streak = win_streak(ta, ta_all)
    tb_streak = win_streak(tb, tb_all)

    # ── 14: Toss winner is team_a ─────────────────────────────────────────
    toss_winner_is_ta = int(match_row['toss_winner'] == ta)

    # ── 15: Toss decision is bat ──────────────────────────────────────────
    toss_decision_bat = int(match_row['toss_decision'] == 'bat')

    # ── 16: Venue bat-first win rate (historical) ─────────────────────────
    venue_matches = past_matches[past_matches['venue'] == venue]
    # team_a = batted first, so team_a winning = bat first won
    venue_batfirst_wr = safe_div(
        (venue_matches['winner'] == venue_matches['team_a']).sum(),
        len(venue_matches)
    )

    # ── 17-18: Home ground flags ──────────────────────────────────────────
    HOME_GROUNDS = {
        'Mumbai Indians'            : ['Wankhede Stadium'],
        'Chennai Super Kings'       : ['MA Chidambaram Stadium'],
        'Royal Challengers Bengaluru': ['M Chinnaswamy Stadium'],
        'Kolkata Knight Riders'     : ['Eden Gardens'],
        'Delhi Capitals'            : ['Arun Jaitley Stadium', 'Feroz Shah Kotla'],
        'Rajasthan Royals'          : ['Sawai Mansingh Stadium'],
        'Punjab Kings'              : ['Punjab Cricket Association IS Bindra Stadium', 'IS Bindra Stadium'],
        'Sunrisers Hyderabad'       : ['Rajiv Gandhi International Stadium'],
        'Gujarat Titans'            : ['Narendra Modi Stadium', 'Motera Stadium'],
        'Lucknow Super Giants'      : ['BRSABV Ekana Cricket Stadium'],
    }
    ta_home = int(venue in HOME_GROUNDS.get(ta, []))
    tb_home = int(venue in HOME_GROUNDS.get(tb, []))

    # ── 19-20: Season win rates ───────────────────────────────────────────
    season_matches = past_matches[past_matches['season_year'] == season]
    ta_season = season_matches[
        (season_matches['team_a'] == ta) | (season_matches['team_b'] == ta)
    ]
    tb_season = season_matches[
        (season_matches['team_a'] == tb) | (season_matches['team_b'] == tb)
    ]
    ta_season_wr = safe_div(team_wins(ta, ta_season), len(ta_season))
    tb_season_wr = safe_div(team_wins(tb, tb_season), len(tb_season))

    # ── 21: Match number in season ────────────────────────────────────────
    match_num = len(season_matches) + 1   # +1 because current match isn't in past yet

    # ── 22: Is playoff ────────────────────────────────────────────────────
    is_playoff = match_row['is_playoff']

    # ── 23: Day/night flag ────────────────────────────────────────────────
    # Most IPL matches are day/night — check if 'day' column exists
    is_day_night = int(match_row.get('day', 0) != 1) if 'day' in match_row else 1

    # ── 24: Venue average first innings score ─────────────────────────────
    venue_avg_score = 0
    if len(venue_matches) > 0 and 'team_runs' in balls.columns:
        venue_ball_past = balls[
            (balls['venue'] == venue) &
            (balls['date'] < match_row['date']) &
            (balls['innings'] == 1)
        ]
        if len(venue_ball_past) > 0:
            first_inn_scores = venue_ball_past.groupby('match_id')['runs_total'].sum()
            venue_avg_score = first_inn_scores.mean()

    return {
        'match_id'            : match_row['match_id'],
        'date'                : match_row['date'],
        'season_year'         : season,
        'team_a'              : ta,
        'team_b'              : tb,
        'venue'               : venue,
        # Historical win rates
        'ta_overall_wr'       : round(ta_overall_wr, 4),
        'tb_overall_wr'       : round(tb_overall_wr, 4),
        'ta_venue_wr'         : round(ta_venue_wr, 4),
        'tb_venue_wr'         : round(tb_venue_wr, 4),
        'ta_h2h_wr'           : round(ta_h2h_wr, 4),
        'ta_batfirst_wr'      : round(ta_batfirst_wr, 4),
        'tb_batfirst_wr'      : round(tb_batfirst_wr, 4),
        # Recent form
        'ta_last5_wr'         : round(ta_last5_wr, 4),
        'tb_last5_wr'         : round(tb_last5_wr, 4),
        'ta_last5_margin'     : round(ta_last5_margin, 2),
        'tb_last5_margin'     : round(tb_last5_margin, 2),
        'ta_streak'           : ta_streak,
        'tb_streak'           : tb_streak,
        # Toss & venue
        'toss_winner_is_ta'   : toss_winner_is_ta,
        'toss_decision_bat'   : toss_decision_bat,
        'venue_batfirst_wr'   : round(venue_batfirst_wr, 4),
        'ta_home'             : ta_home,
        'tb_home'             : tb_home,
        # Season context
        'ta_season_wr'        : round(ta_season_wr, 4),
        'tb_season_wr'        : round(tb_season_wr, 4),
        'match_num_in_season' : match_num,
        'is_playoff'          : int(is_playoff),
        # Conditions
        'is_day_night'        : is_day_night,
        'venue_avg_first_inn_score': round(venue_avg_score, 2),
        # Target
        'team_a_won'          : match_row['team_a_won'],
    }

## 5. Run the Feature Loop

For each match, pass all PREVIOUS matches as `past_matches` and compute features.
This takes 2-3 minutes — it's doing 1,169 lookups.

In [ ]:
from tqdm.notebook import tqdm   # progress bar — install with: pip install tqdm

rows = []

for i, match_row in tqdm(matches.iterrows(), total=len(matches), desc='Computing features'):
    # past_matches = all matches with date strictly before current match
    past = matches[matches['date'] < match_row['date']]
    rows.append(compute_team_features(match_row, past))

team_features = pd.DataFrame(rows)
print(f'\nDone! Shape: {team_features.shape}')
team_features.head()

## 6. Data Leakage Validation

Pick 5 matches from early 2024 (test set boundary) and manually verify
that features only reflect past data.

In [ ]:
# Pick 5 matches from March 2024
boundary_matches = team_features[
    (team_features['date'] >= '2024-03-01') &
    (team_features['date'] <= '2024-03-31')
].head(5)

print('Boundary check — features for early 2024 matches:')
print('These win rates should only reflect data up to late 2023')
print(boundary_matches[['date','team_a','team_b','ta_overall_wr','tb_overall_wr','ta_season_wr']].to_string(index=False))

In [ ]:
# Sanity checks
print('=== SANITY CHECKS ===')

# 1. First match should have all 0.5 defaults (no history yet)
first = team_features.iloc[0]
print(f'First match ({first["date"].date()}) overall win rates: ta={first["ta_overall_wr"]}, tb={first["tb_overall_wr"]} (expect 0.5)')

# 2. CSK at Chepauk should have high venue win rate
csk_chepauk = team_features[
    (team_features['team_a'] == 'Chennai Super Kings') &
    (team_features['venue'].str.contains('Chidambaram', na=False))
].tail(5)
if len(csk_chepauk) > 0:
    print(f'CSK at Chepauk — avg venue win rate: {csk_chepauk["ta_venue_wr"].mean():.3f} (expect high, >0.55)')

# 3. team_a_won should be roughly 50%
print(f'team_a_won rate: {team_features["team_a_won"].mean():.3f} (expect ~0.5)')

# 4. No nulls in feature columns
null_check = team_features.isnull().sum()
print(f'Null values in features: {null_check[null_check > 0].to_dict()}')

## 7. Save

In [ ]:
team_features.to_csv('../data/processed/team_features.csv', index=False)
print(f'Saved team_features.csv — {len(team_features):,} rows, {team_features.shape[1]} columns')
print(f'Feature columns: {[c for c in team_features.columns if c not in ["match_id","date","season_year","team_a","team_b","venue","team_a_won"]]}')

## Your Tasks (Before Phase 4)

```
□ For EACH feature, write one sentence explaining what it captures:
  e.g. ta_h2h_wr → 'How often team_a beats team_b historically'

□ Pick 3 matches you remember watching — check if the computed
  features make sense:
  e.g. CSK at Chepauk should have high ta_venue_wr
       MI in a final should have high ta_overall_wr

□ Modify: add one new feature yourself — ideas:
  - days_since_last_match (rest/fatigue proxy)
  - ta_chase_win_rate (how often team_a wins when chasing)

□ Add to interview notebook:
  Q: 'What is data leakage? How did you prevent it?'
  A: [your answer]

  Q: 'Which feature do you think is most predictive? Why?'
  A: [your answer]

  Q: 'Why did you compute features using only past data?'
  A: [your answer]
```